# Plotting

In this notebook we explore how to plot wells and fits with `gwrefpy`.

This notebook can be downloaded from the source code [here](https://github.com/sgfsweden/gwrefpy/blob/main/docs/user_guide/4_plotting.ipynb).

`gwrefpy` draws its plots with matplotlib:

- Every well and every fit has a `plot()` method. A fit also has `fit.plots.diagnostics()`, a cross-plot of the two wells.
- Plots that combine several wells or fits live in the `gwrefpy.plots` module, e.g. `gr.plots.compare()`.

Every plot takes the `tmin` and `tmax` arguments to only show data inside a time window, draws on `ax` when given (otherwise on a new figure of size `figsize`), and returns the matplotlib `Axes`. `gwrefpy` has no plot styles of its own: titles, labels, colours and saving are done with matplotlib on the returned `Axes`.

```{tip}
If you are not within a Jupyter notebook, call `plt.show()` to show the plots in a separate window.
```

In [ ]:
import gwrefpy as gr
import matplotlib.pyplot as plt

gr.set_log_level("ERROR")

model = gr.Model("small_example.gwref")
model

This example model contains two observation wells, two reference wells and two fits. We replace the fits with two new ones that fit the observation well _12GIPGW_ to each of the reference wells. This gives the fits readable names and two candidate fits to compare later on.

In [ ]:
model.fits.clear()
model.fit("12GIPGW", model.wells.ref(), offset="0D", tmax="2020-03-23", report=False)

## Plotting wells

A well's `plot()` method draws its timeseries.

```{tip}
Appending a `;` to the last line of a cell, or assigning the result to `_`, hides the returned `Axes` so that only the plot is shown.
```

In [ ]:
model.wells["12GIPGW"].plot();

Use `tmin` and `tmax` to zoom into a period. A partial date like `"2020-03"` includes the whole month.

In [ ]:
model.wells["12GIPGW"].plot(tmin="2020-02", tmax="2020-03");

To draw several wells in one plot, pass them to `gr.plots.compare()`, either as a list or as a collection such as `model.wells`, or `model.wells.ref()` for only the reference wells.

In [ ]:
gr.plots.compare(model.wells);

## Plotting fits

A fit's `plot()` method draws the observation well heads, the fitted heads (the reference well heads transformed by the regression) and the prediction interval around them. Let's get the fit to the reference well _45LOGW_ and plot it.

In [ ]:
fit = model.fits.get(ref="45LOGW")
fit.plot();

Three arguments add more to the plot:

- `outliers=True` marks the observations outside the prediction interval.
- `calibration_period=True` shades the period the fit was calibrated on.
- `ref=True` also draws the reference well's own heads, before the regression transforms them. They are often at a different level than the observation well, which stretches the y-axis, so we leave it out here.

In [ ]:
fit.plot(outliers=True, calibration_period=True);

## Diagnostics

`fit.plots.diagnostics()` draws a cross-plot of the two wells: reference well heads on the x-axis and observation well heads on the y-axis. Filled points are inside the calibration period and hollow points are outside it. The solid line is the regression and the dashed lines are the prediction interval. The legend shows the method, the number of data points, the RMSE and the R² of the fit.

In [ ]:
fit.plots.diagnostics();

Pass `stats=False` to only show the fit's name in the legend. Like the other plots, `tmin` and `tmax` select which points are shown.

In [ ]:
fit.plots.diagnostics(stats=False, tmin="2020-03");

## Comparing fits

`gr.plots.compare()` also takes fits, as a list or a collection such as `model.fits`. It draws all their cross-plots on one `Axes`, each fit in its own colour and marker. This is useful to compare candidate reference wells for the same observation well.

In [ ]:
gr.plots.compare(model.fits);

`compare()` takes the same `tmin`, `tmax` and `stats` arguments as `diagnostics()`. A list of fits and wells mixed together raises an error.

`compare()` is also useful to compare fitting methods. Fits made from exactly the same points, e.g. different methods fitted to the same wells and calibration period, share their points: the points are drawn once in grey, and each fit's curve and prediction interval gets its own colour. Here we compare our linear fit with a second-degree polynomial fit (see the [fitting methods notebook](7_fitting_methods.ipynb)):

In [ ]:
npoly = model.fit(
    "12GIPGW", "45LOGW", offset="0D", tmax="2020-03-23", method=gr.NPoly(degree=2), report=False
)
gr.plots.compare([fit, npoly]);

## Combining and customising plots

Every plot draws on the `Axes` passed as `ax`, so several plots can be combined in one figure. The returned `Axes` can be changed with matplotlib as usual, for example to add titles or to save the figure.

In [ ]:
fig, (top, bottom) = plt.subplots(nrows=2, figsize=(7, 8), layout="constrained")

fit.plot(calibration_period=True, ax=top)
fit.plots.diagnostics(ax=bottom)

top.set_title("Fitted heads")
bottom.set_title("Diagnostics");

# fig.savefig("12GIPGW_45LOGW.png", dpi=200)

A loop over a collection of fits gives one plot per fit. Here we loop over the two linear fits:

In [ ]:
fits = model.fits.filter(method="linear")
fig, axs = plt.subplots(ncols=len(fits), sharey=True, figsize=(10, 4), layout="constrained")
for ax, candidate in zip(axs, fits):
    candidate.plots.diagnostics(ax=ax, stats=False)

To change the look of all plots at once, use a [matplotlib style sheet](https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html), e.g. `plt.style.use("ggplot")`.

## Migrating from the old plotting methods

Earlier versions of `gwrefpy` plotted from the model, with methods such as `model.plot_fits()`. These methods still work but are deprecated and will be removed in version 2.0. The table below lists what to use instead.

| Old | New |
| --- | --- |
| `model.plot_wells()` | `gr.plots.compare(model.wells)` |
| `model.plot_wells(wells=well)` | `well.plot()` |
| `model.plot_fits(fit)` | `fit.plot()` |
| `model.plot_fits(..., plot_separately=True)` | a loop over the fits with `fit.plot(ax=...)` |
| `mark_outliers=True` | `outliers=True` |
| `show_initiation_period=True` | `calibration_period=True` |
| `plot_ref_well=True` | `ref=True` |
| `model.plot_fitmethod(fit)` | `fit.plots.diagnostics()` |
| `model.show_plots()` | `plt.show()` |
| `title`, `xlabel`, `ylabel`, `save_path` | matplotlib on the returned `Axes`, e.g. `ax.set_title()` and `ax.figure.savefig()` |
| `plot_style`, `color_style`, `offset_text`, `num` | removed; use matplotlib instead |